In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np

C:\Users\z3258367\AppData\Local\Temp\ipykernel_22564\1438236441.py:2: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


In [79]:
folder = "C:\\Users\\z3258367\\OneDrive - UNSW\\#PhD\\Walkability\\Other Cities\\Colouring data & results\\"
data = "C:\\Users\\z3258367\\OneDrive - UNSW\\#PhD\\Data\\"
meshblocks = pd.read_csv(''.join(folder + "Shared Aus Data\\MB_DZN_SA2_2016_AUST.csv"))
SA1s = gpd.read_file(''.join(data + "ABS Data\\2016_SA1_shape\\SA1_2016_AUST.shp"))
DZNs = pd.read_csv(''.join(folder + "Sydney Data\\Data\\2016 NSW DZN employment.csv"), dtype='int64')
mb_shapes = gpd.read_file(''.join(data + "ABS Data\\MBs\\2016_NSW_MBs\\MB_2016_NSW.shp"))

Select only employment-generating meshblocks. 'Other' is typically a designation for meshblocks with a mixture of land uses. Usually these are larger semi-rural meshblocks that will not be significant for walkability results either way. We thought it more accurate to include these meshblocks.

In [5]:
employ_mbs = meshblocks[meshblocks['MB_CATEGORY_NAME_2016'].isin(
    ['Commercial','Primary Production','Hospital/Medical','Education','Other','Industrial'])]

Next sum the meshblock areas by DZN code. Ie, the output is a list of DZNs along with summed areas for each of them, of the area of the employment meshbocks within. This is then joined to the DZN Place of Work numbers data.

In [6]:
employ_areas = pd.DataFrame(employ_mbs.groupby('DZN_CODE_2016')['AREA_ALBERS_SQKM'].sum())

In [7]:
DZN_areas = DZNs.join(employ_areas, on='DZN (POW)', how='left')

This is the portion of jobs we lose with this method - jobs that are in DZNs that are made of entirely excluded meshblocks (residential, transport, parkland, water). For states I have done so far it's under 5% so considered it acceptable. One potential improvement would be to manually change some meshblock categories, for example an airport from 'transport' to 'industrial'. (Most transport meshblocks are just road or rail corridors so are better excluded).

In [10]:
# sometimes 'Count', sometimes 'Number' or 'Jobs'
DZN_areas[(DZN_areas['Number']>0) & (DZN_areas['AREA_ALBERS_SQKM'].isna())]['Number'].sum()/DZN_areas['Number'].sum()

0.04737342269269473

The DZN 'Job Density' is the number of people who report that DZN as their place of work, divided by the area of employment meshblocks within. This density is then used to calculate the job number for each of those meshblocks.

In [11]:
DZN_areas['JobDensity'] = DZN_areas['Number']/DZN_areas['AREA_ALBERS_SQKM']

employ_mbs = employ_mbs.join(DZN_areas.set_index('DZN (POW)'), on='DZN_CODE_2016', how='inner', rsuffix='_DZN')

employ_mbs['Jobs'] = employ_mbs['JobDensity']*employ_mbs['AREA_ALBERS_SQKM']

The employment figures are attached to the meshblock shapefiles, and centroids are also output, as currently I am using the centroids as the points for walkability calculations.

In [12]:
mb_shapes['MB_CODE16'] = mb_shapes['MB_CODE16'].astype('int64')
employ_mbs['MB_CODE_2016'] = employ_mbs['MB_CODE_2016'].astype('int64')

employ_shapes = mb_shapes.join(employ_mbs.set_index('MB_CODE_2016')[['DZN_CODE_2016','Jobs']], how='right', on='MB_CODE16')

In [45]:
centroids = employ_shapes.copy()
centroids.geometry = (centroids.geometry
                         .to_crs('EPSG:7856')
                         .centroid)

In [77]:
# SA1 addition
numbers = centroids[centroids['AREASQKM16']>0].groupby('SA1_MAIN16')['MB_CODE16'].count()
areas = centroids[centroids['AREASQKM16']>0].groupby('SA1_MAIN16')['AREASQKM16'].mean()
jobs = centroids[centroids['AREASQKM16']>0].groupby('SA1_MAIN16')['Jobs'].sum()
max_jobs = centroids[centroids['AREASQKM16']>0].groupby('SA1_MAIN16')['Jobs'].idxmax()
# want to place the new points not at the centroids of SA1s, but on the meshblock with the most jobs within them.
max_jobs = centroids.loc[max_jobs][['SA1_MAIN16','Jobs', 'geometry']].set_index('SA1_MAIN16')
numbers[ratio>15][numbers>1].sum()

6685

In [80]:
SA1s = SA1s.set_index('SA1_MAIN16')

In [81]:
#SA1s where index is in ratio index
Sydney_SA1s = SA1s[SA1s.index.isin(ratio.index)]

In [82]:
small_SA1s = Sydney_SA1s[ratio>15][numbers>1]

c:\Users\z3258367\Anaconda3\envs\ox_ua\lib\site-packages\geopandas\geodataframe.py:1415: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)


In [86]:
# add Jobs to small SA1s
small_SA1s['Jobs'] = jobs
small_SA1s['prev_mb'] = numbers
small_SA1s = small_SA1s.rename(columns={'SA1_7DIGIT':'SA1_7DIG16', 'STATE_NAME':'STE_NAME16', 'STATE_CODE':'STE_CODE16', 'AREA_SQKM':'AREASQKM16'})
SA1_centroids = small_SA1s.copy()
SA1_centroids.geometry = max_jobs.geometry
SA1_centroids = SA1_centroids.copy().reset_index()


In [85]:
max_jobs

,Jobs,geometry
SA1_MAIN16,,
10102100701,11.748811,POINT (217603.290 6102510.878)
10102100702,192.686407,POINT (213728.806 6077192.527)
10102100704,0.076326,POINT (209365.931 6073372.531)
10102100706,4.634756,POINT (205778.020 6060034.895)
10102100707,6.840673,POINT (213604.256 6063171.691)
...,...,...
12802160732,1043.218759,POINT (316043.774 6228868.632)
12802160734,443.000000,POINT (315661.016 6229450.160)
12802160805,270.805405,POINT (317216.144 6229173.784)


In [87]:
SA1_centroids

,SA1_MAIN16,SA1_7DIG16,STE_CODE16,STE_NAME16,AREASQKM16,geometry,Jobs,prev_mb
0,10102100704,1100704,1,New South Wales,1.2816,POINT (209365.931 6073372.531),0.099514,2
1,10102100917,1100917,1,New South Wales,0.2596,POINT (157935.597 6081220.948),774.084610,3
2,10102100919,1100919,1,New South Wales,0.3559,POINT (157619.175 6081294.481),527.894404,3
3,10102100920,1100920,1,New South Wales,0.0786,POINT (157226.398 6081989.584),164.327448,3
4,10102100924,1100924,1,New South Wales,0.1573,POINT (157812.901 6081421.963),946.839107,4
...,...,...,...,...,...,...,...,...
1685,12802153843,1153843,1,New South Wales,0.1639,POINT (320531.942 6232573.098),704.000000,3
1686,12802153851,1153851,1,New South Wales,0.1359,POINT (321093.506 6232387.612),2744.379075,9
1687,12802160701,1160701,1,New South Wales,0.1262,POINT (316881.298 6228782.065),189.922791,2
1688,12802160708,1160708,1,New South Wales,0.0875,POINT (316714.281 6228804.356),301.163283,4


In [88]:
new_centroids = centroids[~centroids['SA1_MAIN16'].isin(small_SA1s.index)]
new_centroids = pd.concat([new_centroids, SA1_centroids])

In [89]:
new_centroids

,MB_CODE16,MB_CAT16,SA1_MAIN16,SA1_7DIG16,SA2_MAIN16,SA2_5DIG16,SA2_NAME16,SA3_CODE16,SA3_NAME16,SA4_CODE16,SA4_NAME16,GCC_CODE16,GCC_NAME16,STE_CODE16,STE_NAME16,AREASQKM16,geometry,DZN_CODE_2016,Jobs,prev_mb
410,1.000391e+10,Commercial,10901117611,1117611,109011176,11176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,0.0801,POINT (-46117.247 5993875.998),111768918.0,483.458495,NaN
519,1.000469e+10,Education,10901117619,1117619,109011176,11176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,0.0479,POINT (-45483.324 5993486.128),111768918.0,289.109387,NaN
534,1.000485e+10,Commercial,10901117629,1117629,109011176,11176,Lavington,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,0.0376,POINT (-46567.851 5993269.927),111768918.0,226.941815,NaN
25,1.000018e+10,Other,10901117301,1117301,109011173,11173,Albury - North,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,3.3217,POINT (-50789.813 5993187.856),111738912.0,95.895604,NaN
26,1.000018e+10,Primary Production,10901117323,1117323,109011173,11173,Albury - North,10901,Albury,109,Murray,1RNSW,Rest of NSW,1,New South Wales,0.4239,POINT (-49197.138 5993360.653),111738912.0,12.237754,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1685,NaN,NaN,12802153843,1153843,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,New South Wales,0.1639,POINT (320531.942 6232573.098),NaN,704.000000,3.0
1686,NaN,NaN,12802153851,1153851,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,New South Wales,0.1359,POINT (321093.506 6232387.612),NaN,2744.379075,9.0
1687,NaN,NaN,12802160701,1160701,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,New South Wales,0.1262,POINT (316881.298 6228782.065),NaN,189.922791,2.0
1688,NaN,NaN,12802160708,1160708,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,New South Wales,0.0875,POINT (316714.281 6228804.356),NaN,301.163283,4.0


### Export

In [90]:
new_centroids.to_file(''.join(folder + "Shared Aus Data\\NSW_Employment_MB_SA1s.gpkg"), layer='centroids')